# 02 数据清洗 (Data Cleaning)

**课程**: 数据分析  
**姓名**: 廖婉琼  
**学号**: 25210178  

---

本 Notebook 完成以下清洗任务：
1. 缺失值检测与处理
2. 日期格式统一
3. 数据类型检查与修正
4. 重复值检测与处理
5. 离群值标注
6. 宽表与长表转换
7. 多表合并
8. 进阶存储（SQLite）

In [1]:
import pandas as pd
import numpy as np
import os
import sqlite3
import warnings
warnings.filterwarnings('ignore')

# ============ 路径配置 ============
BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
DATA_DIR = os.path.join(BASE_DIR, 'data')
STOCK_DIR = os.path.join(DATA_DIR, 'stock')
INDEX_DIR = os.path.join(DATA_DIR, 'index')
MACRO_DIR = os.path.join(DATA_DIR, 'macro')
FINANCE_DIR = os.path.join(DATA_DIR, 'finance')
CLEAN_DIR = os.path.join(DATA_DIR, 'clean')
COMBINED_DIR = os.path.join(DATA_DIR, 'combined')
DB_PATH = os.path.join(DATA_DIR, 'clean', 'finance.db')

print('数据清洗环境初始化完成')

数据清洗环境初始化完成


## 2.1 加载原始数据

In [2]:
stock_list = {
    '002594': '比亚迪', '601633': '长城汽车', '600519': '贵州茅台',
    '000858': '五粮液', '601857': '中国石油', '601088': '中国神华',
    '600941': '中国移动', '000063': '中兴通讯', '002352': '顺丰控股', '600233': '圆通速递'
}

stock_dfs = {}
for code, name in stock_list.items():
    fpath = os.path.join(STOCK_DIR, f'stock_{code}.csv')
    if os.path.exists(fpath):
        stock_dfs[name] = pd.read_csv(fpath)
        print(f'  OK {name}({code}): {stock_dfs[name].shape}')
    else:
        print(f'  MISSING {name}({code})')

print(f'\n共加载 {len(stock_dfs)} 只股票数据')

  OK 比亚迪(002594): (1544, 9)
  OK 长城汽车(601633): (1544, 9)
  OK 贵州茅台(600519): (1544, 9)
  OK 五粮液(000858): (1544, 9)
  OK 中国石油(601857): (1544, 9)
  OK 中国神华(601088): (1544, 9)
  OK 中国移动(600941): (1057, 9)
  OK 中兴通讯(000063): (1544, 9)
  OK 顺丰控股(002352): (1544, 9)
  OK 圆通速递(600233): (1544, 9)

共加载 10 只股票数据


In [3]:
index_dfs = {}
for code, name in [('000300', '沪深300'), ('000001', '上证综指')]:
    fpath = os.path.join(INDEX_DIR, f'index_{code}.csv')
    if os.path.exists(fpath):
        index_dfs[name] = pd.read_csv(fpath)
        print(f'  OK {name}({code}): {index_dfs[name].shape}')

macro_dfs = {}
for fname in sorted(os.listdir(MACRO_DIR)):
    if fname.endswith('.csv'):
        key = fname.replace('macro_', '').replace('.csv', '')
        macro_dfs[key] = pd.read_csv(os.path.join(MACRO_DIR, fname))
        print(f'  OK macro_{key}: {macro_dfs[key].shape}')

finance_df = pd.read_csv(os.path.join(FINANCE_DIR, 'finance_ratios.csv'))
print(f'  OK finance_ratios: {finance_df.shape}')

  OK 沪深300(000300): (1544, 9)
  OK 上证综指(000001): (1544, 9)
  OK macro_cpi: (68, 3)
  OK macro_industrial: (61, 3)
  OK macro_lpr: (77, 3)
  OK macro_m2: (76, 3)
  OK macro_usd_cny: (9, 3)
  OK finance_ratios: (200, 5)


## 2.2 缺失值检测与处理

In [4]:
print('='*60)
print('2.2 缺失值检测')
print('='*60)

for name, df in stock_dfs.items():
    missing = df.isnull().sum()
    total = missing.sum()
    status = f'{total} 个缺失' if total > 0 else '无缺失值'
    print(f'{name}: {status}')

for name, df in index_dfs.items():
    total = df.isnull().sum().sum()
    print(f'{name}: {total} 个缺失' if total > 0 else f'{name}: 无缺失值')

for key, df in macro_dfs.items():
    total = df.isnull().sum().sum()
    print(f'{key}: {total} 个缺失' if total > 0 else f'{key}: 无缺失值')

fm = finance_df.isnull().sum().sum()
if fm > 0:
    finance_df['value'] = finance_df['value'].fillna(method='ffill')
    print(f'finance: {fm} 个缺失, 已前向填充')
else:
    print('finance: 无缺失值')

2.2 缺失值检测
比亚迪: 无缺失值
长城汽车: 无缺失值
贵州茅台: 无缺失值
五粮液: 无缺失值
中国石油: 无缺失值
中国神华: 20 个缺失
中国移动: 无缺失值
中兴通讯: 无缺失值
顺丰控股: 无缺失值
圆通速递: 无缺失值
沪深300: 无缺失值
上证综指: 无缺失值
cpi: 无缺失值
industrial: 无缺失值
lpr: 无缺失值
m2: 无缺失值
usd_cny: 无缺失值
finance: 无缺失值


## 2.3 日期格式统一

In [5]:
print('='*60)
print('2.3 日期格式统一')
print('='*60)

for name, df in stock_dfs.items():
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    print(f'{name}: {df["date"].dtype}')

for name, df in index_dfs.items():
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    print(f'{name}: {df["date"].dtype}')

for key, df in macro_dfs.items():
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    print(f'{key}: {df["date"].dtype}')

print('所有日期已统一为 datetime64')

2.3 日期格式统一
比亚迪: datetime64[ns]
长城汽车: datetime64[ns]
贵州茅台: datetime64[ns]
五粮液: datetime64[ns]
中国石油: datetime64[ns]
中国神华: datetime64[ns]
中国移动: datetime64[ns]
中兴通讯: datetime64[ns]
顺丰控股: datetime64[ns]
圆通速递: datetime64[ns]
沪深300: datetime64[ns]
上证综指: datetime64[ns]
cpi: datetime64[ns]
industrial: datetime64[ns]
lpr: datetime64[ns]
m2: datetime64[ns]
usd_cny: datetime64[ns]
所有日期已统一为 datetime64


## 2.4 数据类型检查与修正

In [6]:
print('='*60)
print('2.4 数据类型检查与修正')
print('='*60)

numeric_cols = ['open', 'high', 'low', 'close', 'volume', 'amount']

for name, df in stock_dfs.items():
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    # 从收盘价计算日收益率(%)
    df['daily_return'] = df['close'].pct_change() * 100
    print(f'{name}: 数值列已修正, daily_return已计算')

for name, df in index_dfs.items():
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df['daily_return'] = df['close'].pct_change() * 100
    print(f'{name}: 数值列已修正')

for key, df in macro_dfs.items():
    if 'value' in df.columns:
        df['value'] = pd.to_numeric(df['value'], errors='coerce')
    print(f'{key}: value已修正')

finance_df['value'] = pd.to_numeric(finance_df['value'], errors='coerce')
finance_df['year'] = finance_df['year'].astype(int)
print('finance: 已修正')

2.4 数据类型检查与修正


比亚迪: 数值列已修正, daily_return已计算
长城汽车: 数值列已修正, daily_return已计算
贵州茅台: 数值列已修正, daily_return已计算
五粮液: 数值列已修正, daily_return已计算
中国石油: 数值列已修正, daily_return已计算
中国神华: 数值列已修正, daily_return已计算
中国移动: 数值列已修正, daily_return已计算
中兴通讯: 数值列已修正, daily_return已计算
顺丰控股: 数值列已修正, daily_return已计算
圆通速递: 数值列已修正, daily_return已计算
沪深300: 数值列已修正
上证综指: 数值列已修正
cpi: value已修正
industrial: value已修正
lpr: value已修正
m2: value已修正
usd_cny: value已修正
finance: 已修正


## 2.5 重复值检测与处理

In [7]:
print('='*60)
print('2.5 重复值检测与处理')
print('='*60)

total_dup = 0
for name, df in stock_dfs.items():
    n = df.duplicated(subset=['code', 'date']).sum()
    total_dup += n
    if n > 0:
        df.drop_duplicates(subset=['code', 'date'], keep='first', inplace=True)
        df.reset_index(drop=True, inplace=True)
        print(f'{name}: 删除 {n} 条重复')
    else:
        print(f'{name}: 无重复值')

for name, df in index_dfs.items():
    n = df.duplicated(subset=['code', 'date']).sum()
    total_dup += n
    if n > 0:
        df.drop_duplicates(subset=['code', 'date'], keep='first', inplace=True)
        df.reset_index(drop=True, inplace=True)
        print(f'{name}: 删除 {n} 条重复')
    else:
        print(f'{name}: 无重复值')

for key, df in macro_dfs.items():
    n = df.duplicated().sum()
    total_dup += n
    if n > 0:
        df.drop_duplicates(keep='first', inplace=True)
        df.reset_index(drop=True, inplace=True)
        print(f'{key}: 删除 {n} 条重复')
    else:
        print(f'{key}: 无重复值')

print(f'\n共处理 {total_dup} 条重复记录')

2.5 重复值检测与处理
比亚迪: 无重复值
长城汽车: 无重复值
贵州茅台: 无重复值
五粮液: 无重复值
中国石油: 无重复值
中国神华: 无重复值
中国移动: 无重复值
中兴通讯: 无重复值
顺丰控股: 无重复值
圆通速递: 无重复值
沪深300: 无重复值
上证综指: 无重复值
cpi: 无重复值
industrial: 无重复值
lpr: 无重复值
m2: 无重复值
usd_cny: 无重复值

共处理 0 条重复记录


print('='*60)
print('2.6 离群值标注')
print('='*60)

# 作业要求: 计算日收益率，单日涨跌幅超±20%标为 is_extreme=True
# 注意: 不删除离群值，仅标注
outlier_summary = []
for name, df in stock_dfs.items():
    df['is_extreme'] = False
    extreme_mask = df['daily_return'].abs() > 20  # 单日涨跌幅超±20%
    df.loc[extreme_mask, 'is_extreme'] = True
    n_extreme = extreme_mask.sum()
    outlier_summary.append({
        '股票': name,
        '极端涨跌天数': int(n_extreme),
        '占比': f'{n_extreme/len(df)*100:.2f}%',
        '阈值': '±20%',
        '处理方式': '标注，不删除'
    })

outlier_df = pd.DataFrame(outlier_summary)
print(outlier_df.to_string(index=False))

print('''\n【说明】
- 标注规则: 单日涨跌幅绝对值 > 20% 标记为 is_extreme=True
- 处理方式: 仅标注不删除，保留原始数据完整性
- 可能成因: 涨跌停板（主板±10%、创业板/科创板±20%）、
  除权除息、公司重大事件公告、ST/退市风险警示等
''')


In [8]:
print('='*60)
print('2.6 离群值标注 (IQR 方法)')
print('='*60)

def detect_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    mask = (series < lower) | (series > upper)
    return mask, lower, upper

outlier_summary = []
for name, df in stock_dfs.items():
    data = df['daily_return'].dropna()
    if len(data) == 0:
        continue
    mask, lower, upper = detect_outliers_iqr(data)
    n_outliers = mask.sum()
    df['outlier'] = 0
    df.loc[mask.index[mask], 'outlier'] = 1
    outlier_summary.append({
        '股票': name,
        '离群值数量': int(n_outliers),
        '离群值占比': f'{n_outliers/len(df)*100:.2f}%',
        'IQR下界': f'{lower:.2f}',
        'IQR上界': f'{upper:.2f}'
    })

outlier_df = pd.DataFrame(outlier_summary)
print(outlier_df.to_string(index=False))
print('\n注: 离群值已标注(outlier列)但未删除，保留原始数据完整性')

2.6 离群值标注 (IQR 方法)


  股票  离群值数量 离群值占比 IQR下界 IQR上界
 比亚迪     94 6.09% -5.46  5.46
长城汽车    110 7.12% -5.52  5.35
贵州茅台    116 7.51% -3.20  3.19
 五粮液     96 6.22% -4.35  4.22
中国石油     98 6.35% -3.45  3.53
中国神华     89 5.76% -3.73  3.89
中国移动     59 5.58% -3.03  3.17
中兴通讯     89 5.76% -5.65  5.52
顺丰控股     77 4.99% -4.12  3.97
圆通速递     95 6.15% -4.65  4.46

注: 离群值已标注(outlier列)但未删除，保留原始数据完整性


## 2.7 宽表与长表转换

In [9]:
print('='*60)
print('2.7 宽表与长表转换')
print('='*60)

# 宽表: 每只股票一列收盘价
close_wide = pd.DataFrame()
for name, df in stock_dfs.items():
    close_wide[name] = df.set_index('date')['close']
for name, df in index_dfs.items():
    close_wide[name] = df.set_index('date')['close']
close_wide.index.name = 'date'
close_wide.sort_index(inplace=True)

print(f'宽表 (close_wide): {close_wide.shape}')
print(close_wide.head(3))

# 长表: melt
close_long = close_wide.reset_index().melt(
    id_vars='date', var_name='stock_name', value_name='close'
)
close_long.dropna(subset=['close'], inplace=True)
close_long.sort_values(['stock_name', 'date'], inplace=True)
close_long.reset_index(drop=True, inplace=True)

print(f'\n长表 (close_long): {close_long.shape}')
print(close_long.head(6))

2.7 宽表与长表转换
宽表 (close_wide): (1544, 12)
                  比亚迪      长城汽车        贵州茅台         五粮液      中国石油       中国神华  \
date                                                                           
2020-01-02  15.582369  8.386021  979.045560  111.862382  4.377881  11.610515   
2020-01-03  15.540315  8.358675  934.477327  110.566581  4.437546  11.591778   
2020-01-06  15.617952  8.167255  933.983472  109.423227  4.646371  11.729180   

            中国移动       中兴通讯       顺丰控股       圆通速递     沪深300      上证综指  
date                                                                   
2020-01-02   NaN  32.718117  33.969811  11.588712  4152.240  3085.197  
2020-01-03   NaN  33.733347  33.706971  11.615916  4144.964  3083.785  
2020-01-06   NaN  33.871788  33.398813  11.479898  4129.295  3083.408  

长表 (close_long): (18041, 3)
        date stock_name     close
0 2020-01-02       上证综指  3085.197
1 2020-01-03       上证综指  3083.785
2 2020-01-06       上证综指  3083.408
3 2020-01-07       上证综指  3104.801


## 2.8 多表合并

In [10]:
print('='*60)
print('2.8 多表合并')
print('='*60)

macro_wide = pd.DataFrame()
for key, df in macro_dfs.items():
    for ind in df['indicator'].unique():
        sub = df[df['indicator'] == ind][['date', 'value']].copy()
        col_name = f'{key}_{ind}'
        macro_wide[col_name] = sub.set_index('date')['value']

macro_wide.index.name = 'date'
macro_daily = macro_wide.resample('D').ffill()
combined = close_wide.join(macro_daily, how='left')
combined.index.name = 'date'

print(f'宏观指标宽表: {macro_wide.shape}')
print(f'合并后数据集: {combined.shape}')
print(f'列名: {combined.columns.tolist()}')
print(combined.tail(3))

2.8 多表合并


宏观指标宽表: (68, 5)
合并后数据集: (1544, 17)
列名: ['比亚迪', '长城汽车', '贵州茅台', '五粮液', '中国石油', '中国神华', '中国移动', '中兴通讯', '顺丰控股', '圆通速递', '沪深300', '上证综指', 'cpi_CPI同比增速', 'industrial_工业增加值同比增速', 'lpr_LPR1Y', 'm2_M2同比增速', 'usd_cny_USD_CNY汇率']
              比亚迪   长城汽车    贵州茅台    五粮液   中国石油   中国神华    中国移动   中兴通讯   顺丰控股  \
date                                                                          
2026-05-19  96.00  18.45  1324.3  85.78  11.56  46.07  101.00  37.65  35.29   
2026-05-20  93.36  18.13  1315.0  85.47  11.63  46.00   99.16  37.18  35.72   
2026-05-21  93.51  17.83  1311.0  85.35  11.29  45.26   96.75  34.83  35.48   

             圆通速递      沪深300       上证综指  cpi_CPI同比增速  industrial_工业增加值同比增速  \
date                                                                         
2026-05-19  19.46  4852.8832  4169.5378          NaN                   NaN   
2026-05-20  19.34  4850.7000  4162.1845          NaN                   NaN   
2026-05-21  18.75  4783.0976  4077.2765          NaN                   

In [11]:
close_wide.to_csv(os.path.join(CLEAN_DIR, 'stock_close_wide.csv'), encoding='utf-8-sig')
close_long.to_csv(os.path.join(CLEAN_DIR, 'stock_close_long.csv'), index=False, encoding='utf-8-sig')
combined.to_csv(os.path.join(COMBINED_DIR, 'stock_macro_combined.csv'), encoding='utf-8-sig')
print('宽表、长表、合并表已保存')

宽表、长表、合并表已保存


## 2.9 进阶存储 — 方式C: SQLite

In [12]:
print('='*60)
print('2.9 进阶存储: SQLite 数据库')
print('='*60)

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

conn = sqlite3.connect(DB_PATH)

for name, df in stock_dfs.items():
    tbl = f'stock_{df["code"].iloc[0]}'
    df.to_sql(tbl, conn, if_exists='replace', index=False)
    print(f'  OK {tbl}: {len(df)} 行')

for name, df in index_dfs.items():
    tbl = f'index_{df["code"].iloc[0]}'
    df.to_sql(tbl, conn, if_exists='replace', index=False)
    print(f'  OK {tbl}: {len(df)} 行')

for key, df in macro_dfs.items():
    df.to_sql(f'macro_{key}', conn, if_exists='replace', index=False)
    print(f'  OK macro_{key}: {len(df)} 行')

finance_df.to_sql('finance_ratios', conn, if_exists='replace', index=False)
print(f'  OK finance_ratios: {len(finance_df)} 行')

close_wide.to_sql('stock_close_wide', conn, if_exists='replace', index=True)
print(f'  OK stock_close_wide: {len(close_wide)} 行')

conn.commit()
print(f'\n数据库已保存: {DB_PATH}')

2.9 进阶存储: SQLite 数据库


  OK stock_2594: 1544 行
  OK stock_601633: 1544 行
  OK stock_600519: 1544 行


  OK stock_858: 1544 行
  OK stock_601857: 1544 行
  OK stock_601088: 1544 行
  OK stock_600941: 1057 行
  OK stock_63: 1544 行


  OK stock_2352: 1544 行


  OK stock_600233: 1544 行


  OK index_300: 1544 行
  OK index_1: 1544 行
  OK macro_cpi: 68 行
  OK macro_industrial: 61 行
  OK macro_lpr: 77 行


  OK macro_m2: 76 行
  OK macro_usd_cny: 9 行
  OK finance_ratios: 200 行
  OK stock_close_wide: 1544 行

数据库已保存: C:\Users\Admin1\WorkBuddy\2026-05-22-16-30-37\dshw-p01\data\clean\finance.db


In [13]:
print('\n--- SQLite 查询验证 ---')
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(f'\n数据库中共有 {len(tables)} 张表:')
for t in tables['name']:
    cnt = pd.read_sql(f'SELECT COUNT(*) as cnt FROM [{t}]', conn).iloc[0, 0]
    print(f'  - {t}: {cnt} 行')

print('\n贵州茅台最近5个交易日:')
result = pd.read_sql('SELECT date, open, high, low, close, volume FROM stock_600519 ORDER BY date DESC LIMIT 5', conn)
print(result.to_string(index=False))

print('\n贵州茅台最新收盘价:')
result = pd.read_sql('SELECT name, close FROM stock_600519 ORDER BY date DESC LIMIT 1', conn)
print(result.to_string(index=False))

conn.close()
print('\nSQLite 验证完成')


--- SQLite 查询验证 ---

数据库中共有 19 张表:
  - stock_2594: 1544 行
  - stock_601633: 1544 行
  - stock_600519: 1544 行
  - stock_858: 1544 行
  - stock_601857: 1544 行
  - stock_601088: 1544 行
  - stock_600941: 1057 行
  - stock_63: 1544 行
  - stock_2352: 1544 行
  - stock_600233: 1544 行
  - index_300: 1544 行
  - index_1: 1544 行
  - macro_cpi: 68 行
  - macro_industrial: 61 行
  - macro_lpr: 77 行
  - macro_m2: 76 行
  - macro_usd_cny: 9 行
  - finance_ratios: 200 行
  - stock_close_wide: 1544 行

贵州茅台最近5个交易日:
               date    open    high     low   close  volume
2026-05-21 00:00:00 1312.98 1323.52 1311.00 1311.00 3886838
2026-05-20 00:00:00 1321.00 1332.99 1314.00 1315.00 4748733
2026-05-19 00:00:00 1321.90 1329.99 1318.00 1324.30 4325464
2026-05-18 00:00:00 1336.00 1342.68 1319.61 1323.00 4966097
2026-05-15 00:00:00 1335.15 1339.28 1327.11 1332.95 5818365

贵州茅台最新收盘价:
name  close
贵州茅台 1311.0

SQLite 验证完成


## 2.10 清洗后数据保存

In [14]:
print('='*60)
print('2.10 保存清洗后数据')
print('='*60)

for name, df in stock_dfs.items():
    code = df['code'].iloc[0]
    df.to_csv(os.path.join(CLEAN_DIR, f'stock_{code}_clean.csv'), index=False, encoding='utf-8-sig')
    print(f'  OK {name}')

for name, df in index_dfs.items():
    code = df['code'].iloc[0]
    df.to_csv(os.path.join(CLEAN_DIR, f'index_{code}_clean.csv'), index=False, encoding='utf-8-sig')
    print(f'  OK {name}')

print(f'\n清洗后数据: {CLEAN_DIR}')
print(f'SQLite数据库: {DB_PATH}')
print(f'合并数据: {COMBINED_DIR}')
print('\n数据清洗全部完成!')

2.10 保存清洗后数据


  OK 比亚迪
  OK 长城汽车


  OK 贵州茅台


  OK 五粮液
  OK 中国石油
  OK 中国神华
  OK 中国移动
  OK 中兴通讯
  OK 顺丰控股
  OK 圆通速递


  OK 沪深300
  OK 上证综指

清洗后数据: C:\Users\Admin1\WorkBuddy\2026-05-22-16-30-37\dshw-p01\data\clean
SQLite数据库: C:\Users\Admin1\WorkBuddy\2026-05-22-16-30-37\dshw-p01\data\clean\finance.db
合并数据: C:\Users\Admin1\WorkBuddy\2026-05-22-16-30-37\dshw-p01\data\combined

数据清洗全部完成!
